# Semana 6 — Data Wrangling e qualidade

Vou auditar a tabela `despesas_clean` que já veio no SQLite e repetir, em memória, as etapas de normalização, deduplicação, imputação **quando necessária** e identificação de outliers pelo IQR.

**Limite dos dados:** o SQLite tem três lançamentos na tabela `despesas`, mas sete na `despesas_clean`. O arquivo não guarda o conjunto completo de entradas brutas que gerou os sete registros. Por isso, não é possível afirmar quantos nulos ou duplicados foram removidos na atividade original.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Funciona ao executar o notebook pela raiz do projeto ou pela pasta da semana.
def encontrar_raiz():
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / 'semana_10_app_streamlit' / 'fintech_integrada.db').is_file():
            return p.resolve()
    raise FileNotFoundError('Não encontrei semana_10_app_streamlit/fintech_integrada.db.')

RAIZ = encontrar_raiz()
BANCO = RAIZ / 'semana_10_app_streamlit' / 'fintech_integrada.db'
conexao = sqlite3.connect(f'file:{BANCO}?mode=ro', uri=True)
conexao.execute('PRAGMA foreign_keys=ON')
print('Banco utilizado:', BANCO.name)
print('Integridade:', conexao.execute('PRAGMA integrity_check').fetchone()[0])


Banco utilizado: fintech_integrada.db
Integridade: ok


In [2]:
consulta = """
    SELECT d.id,d.valor,d.data,d.estudante_id,d.municipio_id,
           d.categoria,d.is_outlier,
           e.nome AS estudante,e.matricula,m.nome AS municipio,m.uf_sigla AS uf
    FROM despesas_clean AS d
    INNER JOIN estudantes AS e ON e.id=d.estudante_id
    INNER JOIN municipios AS m ON m.id=d.municipio_id
    ORDER BY d.data,d.id
"""
despesas = pd.read_sql_query(consulta,conexao,parse_dates=['data'])
despesas['is_outlier'] = despesas['is_outlier'].fillna(0).astype(bool)
display(despesas)


,id,valor,data,estudante_id,municipio_id,categoria,is_outlier,estudante,matricula,municipio,uf
0,1,125.9,2026-09-01,1,3500105,alimentação,False,Ana Souza,2026001,Adamantina,SP
1,2,89.5,2026-09-02,2,3300100,transporte,False,Bruno Lima,2026002,Angra dos Reis,RJ
2,3,210.0,2026-09-03,3,3100104,lazer,False,Carla Mendes,2026003,Abadia dos Dourados,MG
3,4,107.7,2026-09-07,1,3500105,alimentação,False,Ana Souza,2026001,Adamantina,SP
4,5,75.0,2026-09-08,1,3500105,nao_informado,False,Ana Souza,2026001,Adamantina,SP
5,6,5000.0,2026-09-09,1,3500105,transporte,True,Ana Souza,2026001,Adamantina,SP
6,7,64.9,2026-09-10,1,3500105,lazer,False,Ana Souza,2026001,Adamantina,SP


## Diagnóstico da tabela tratada (o que está realmente no arquivo)


In [3]:
campos=['id','valor','data','estudante_id','municipio_id','categoria','is_outlier']
auditada=despesas[campos].copy()
quadro=pd.DataFrame({'tipo':auditada.dtypes.astype(str),'nulos':auditada.isna().sum(),
                     'completude_%':(100*auditada.notna().mean()).round(1)})
display(quadro)
print('Duplicatas de ID nesta tabela:',int(auditada.id.duplicated().sum()))
print('Duplicatas exatas nesta tabela:',int(auditada.duplicated().sum()))
print('Categoria explicitamente não informada:',int(auditada.categoria.eq('nao_informado').sum()))


,tipo,nulos,completude_%
id,int64,0,100.0
valor,float64,0,100.0
data,datetime64[ns],0,100.0
estudante_id,int64,0,100.0
municipio_id,int64,0,100.0
categoria,object,0,100.0
is_outlier,bool,0,100.0


Duplicatas de ID nesta tabela: 0
Duplicatas exatas nesta tabela: 0
Categoria explicitamente não informada: 1


## Limpeza reexecutada sem sobrescrever o banco

A imputação pela mediana continua no código, mas só é aplicada se houver algum valor numérico ausente na entrada. Não atribuo retroativamente ao arquivo recebido um procedimento que não consigo comprovar.


In [4]:
import sys
sys.path.insert(0,str(RAIZ/'semana_10_app_streamlit'))
from fintech_core.qualidade import sanear_copia, auditar

limpo,relatorio=sanear_copia(auditada)
display(pd.Series(relatorio,name='Resultado desta execução').to_frame())
display(limpo)
assert len(limpo)==len(auditada)
assert not limpo['id'].duplicated().any()
assert not limpo['valor'].isna().any()


,Resultado desta execução
duplicados_na_entrada,0
nulos_na_entrada,0
valores_imputados_nesta_execucao,0
linhas_resultantes,7
outliers,1


,id,valor,data,estudante_id,municipio_id,categoria,is_outlier
0,1,125.9,2026-09-01,1,3500105,alimentação,0
1,2,89.5,2026-09-02,2,3300100,transporte,0
2,3,210.0,2026-09-03,3,3100104,lazer,0
3,4,107.7,2026-09-07,1,3500105,alimentação,0
4,5,75.0,2026-09-08,1,3500105,nao_informado,0
5,6,5000.0,2026-09-09,1,3500105,transporte,1
6,7,64.9,2026-09-10,1,3500105,lazer,0


## IQR e conferência das anomalias

Uso o primeiro e terceiro quartis da própria tabela tratada. Um valor fora da faixa `[Q1 − 1,5 × IQR, Q3 + 1,5 × IQR]` é **sinalizado**, não apagado.


In [5]:
q1,q3=limpo.valor.quantile([.25,.75])
iqr=q3-q1
inferior,superior=q1-1.5*iqr,q3+1.5*iqr
conferencia=limpo[['id','valor','is_outlier']].copy()
conferencia['flag_gravada']=auditada.set_index('id').loc[conferencia.id,'is_outlier'].to_numpy()
conferencia['divergente']=conferencia.is_outlier.ne(conferencia.flag_gravada.astype(int))
display(conferencia)
print(f'Q1 = R$ {q1:.2f}; Q3 = R$ {q3:.2f}; IQR = R$ {iqr:.2f}')
print(f'Limites do IQR: R$ {inferior:.2f} a R$ {superior:.2f}')
assert int(limpo.is_outlier.sum())==1


,id,valor,is_outlier,flag_gravada,divergente
0,1,125.9,0,False,False
1,2,89.5,0,False,False
2,3,210.0,0,False,False
3,4,107.7,0,False,False
4,5,75.0,0,False,False
5,6,5000.0,1,True,False
6,7,64.9,0,False,False


Q1 = R$ 82.25; Q3 = R$ 167.95; IQR = R$ 85.70
Limites do IQR: R$ -46.30 a R$ 296.50


## Relação com os lançamentos da tabela de origem

Comparo cada uma das três linhas de `despesas` com as linhas tratadas pelo ID, valor e referência ao estudante e município. As quatro linhas adicionais não aparecem na origem disponibilizada, e não devo inventar essa proveniência.


In [6]:
origem=pd.read_sql_query('SELECT * FROM despesas ORDER BY id',conexao)
confronto=origem.merge(limpo,on='id',how='outer',indicator=True,suffixes=('_origem','_tratada'))
print('Linhas que aparecem apenas em despesas_clean:',int(confronto['_merge'].eq('right_only').sum()))
print('Linhas que aparecem nas duas tabelas:',int(confronto['_merge'].eq('both').sum()))
comuns=confronto.loc[confronto['_merge']=='both']
for campo in ('valor','estudante_id','municipio_id'):
    assert (comuns[f'{campo}_origem']==comuns[f'{campo}_tratada']).all(),campo
display(confronto[['id','valor_origem','valor_tratada','_merge']])
conexao.close()


Linhas que aparecem apenas em despesas_clean: 4
Linhas que aparecem nas duas tabelas: 3


,id,valor_origem,valor_tratada,_merge
0,1,125.9,125.9,both
1,2,89.5,89.5,both
2,3,210.0,210.0,both
3,4,NaN,107.7,right_only
4,5,NaN,75.0,right_only
5,6,NaN,5000.0,right_only
6,7,NaN,64.9,right_only


## Conclusão

A base tratada está sem nulos numéricos e sem IDs duplicados; há uma categoria `nao_informado` e um gasto atípico de R$ 5.000,00. O outlier é relevante para análise de dispersão e treinamento, mas deve ser mantido para mostrar o comportamento da amostra. As contagens de limpeza representam apenas esta auditoria, não a quantidade histórica de erros removidos.
